# Tutorial 1: Quick Start with synthetictext

This notebook walks through the basics of generating synthetic text classification data using the `synthetictext` library.

By the end of this tutorial you will know how to:
1. Define a classification task with `TaskSpec`
2. Generate synthetic samples using `SyntheticDataGenerator`
3. Inspect and save the results

## Installation

Install the library with the OpenAI provider (the default backend):

In [ ]:
# Uncomment to install
# !pip install synthetictext[openai]

## Setup

Set your OpenAI API key. The library picks it up from the environment automatically.

In [ ]:
import os

# Option 1: Set it here (not recommended for shared notebooks)
# os.environ["OPENAI_API_KEY"] = "sk-..."

# Option 2: Already exported in your shell (recommended)
assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY before running this notebook"

## Step 1: Define Your Task

A `TaskSpec` describes the classification task you want synthetic data for. At minimum you need a name, labels, and a description.

In [ ]:
from synthetictext import TaskSpec

task = TaskSpec(
    name="Sentiment Analysis",
    labels={0: "negative", 1: "positive"},
    description="Classify product reviews as positive or negative.",
)

print(f"Task: {task.name}")
print(f"Labels: {task.labels}")
print(f"Binary: {task.is_binary}")

## Step 2: Add Rich Label Descriptions

Providing detailed label descriptions leads to higher quality generation -- the LLM gets better guidance on what each class looks like.

In [ ]:
task = TaskSpec(
    name="Sentiment Analysis",
    labels={0: "negative", 1: "positive"},
    description="Classify product reviews as positive or negative.",
    label_descriptions={
        0: (
            "A review expressing dissatisfaction, frustration, or disappointment "
            "with a product. May mention defects, poor quality, bad customer "
            "service, or failed expectations."
        ),
        1: (
            "A review expressing satisfaction, enthusiasm, or approval of a "
            "product. May mention quality, value for money, exceeding expectations, "
            "or recommending to others."
        ),
    },
    text_domain="product review",
    word_count_range=(15, 60),
    topics={
        "electronics": ["smartphone", "laptop", "headphones", "smart watch"],
        "home": ["vacuum cleaner", "coffee maker", "air purifier"],
        "fashion": ["running shoes", "winter jacket", "backpack"],
    },
)

print(f"Random topic: {task.random_topic()}")
print(f"Random topic: {task.random_topic()}")
print(f"Random topic: {task.random_topic()}")

## Step 3: Generate Synthetic Data

Create a `SyntheticDataGenerator` and call `.generate()`. The simplest call uses the `"direct"` strategy, which asks the LLM to write new samples from scratch.

In [ ]:
from synthetictext import SyntheticDataGenerator

generator = SyntheticDataGenerator(
    task=task,
    llm_provider="openai",
    llm_model="gpt-4o-mini",
)

df = generator.generate(
    language="English",
    num_samples=20,
    strategies=["direct"],
    apply_filters=False,  # skip filtering for now to see raw output
)

print(f"Generated {len(df)} samples")
df.head()

## Step 4: Inspect the Results

In [ ]:
print("Label distribution:")
print(df["label"].value_counts())
print()

for label_idx in task.label_indices:
    label_name = task.label_name(label_idx)
    subset = df[df["label"] == label_idx]
    print(f"\n--- {label_name.upper()} samples ---")
    for _, row in subset.head(3).iterrows():
        print(f"  {row['text'][:100]}..." if len(str(row['text'])) > 100 else f"  {row['text']}")

## Step 5: Generate with Contrastive Pairs

Contrastive generation creates two samples on the same topic -- one per class. This sharpens the decision boundary because the only difference is the sentiment, not the subject matter.

In [ ]:
df_mixed = generator.generate(
    language="English",
    num_samples=24,
    strategies=["direct", "contrastive"],
    strategy_weights=[0.6, 0.4],
    apply_filters=False,
)

print(f"Generated {len(df_mixed)} samples")
print(f"\nBy source strategy:")
print(df_mixed["source"].value_counts())
print(f"\nBy label:")
print(df_mixed["label"].value_counts())

## Step 6: Save Your Data

In [ ]:
from synthetictext.utils import save_data

save_data(df_mixed, "sentiment_synthetic.csv")
print("Saved to sentiment_synthetic.csv")

## Step 7: Serialize and Load Task Configs

`TaskSpec` can be converted to/from a dictionary, making it easy to save and share configurations as JSON or YAML.

In [ ]:
import json

config = task.to_dict()
print(json.dumps(config, indent=2))

In [ ]:
# Reconstruct from dict
task_restored = TaskSpec.from_dict(config)
print(f"Restored task: {task_restored.name}")
print(f"Labels: {task_restored.labels}")
print(f"Topics: {list(task_restored.topics.keys())}")

## Summary

In this tutorial you learned how to:
- Define a task with `TaskSpec` including labels, descriptions, topics, and text domain
- Generate synthetic data using `SyntheticDataGenerator` with `"direct"` and `"contrastive"` strategies
- Combine multiple strategies with weighted allocation
- Save and serialize configurations

**Next:** [Tutorial 2: Quality Filtering](./02_quality_filtering.ipynb) covers the built-in quality filters that remove low-quality, duplicate, and label-leaking samples.